## Setup and Connection

In [2]:
import json
from pathlib import Path
from pprint import pprint
from pymongo import MongoClient

## Load the data

In [3]:
# Connect to local MongoDB
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['project']
collection = db['credit_applications']
print("Connected to MongoDB successfully!")

Connected to MongoDB successfully!


In [4]:
# Load the json file
current_dir = Path.cwd()
repo_root = current_dir.parent
json_path = repo_root / "data" / "clean_credit_applications.json"

with open(json_path, 'r') as file:
    data = json.load(file)

# Clear the Collection
collection.delete_many({})

# Insert the data into a Collection
try:
    collection.insert_many(data)
    print(f"Successfully inserted {len(data)} documents.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully inserted 500 documents.


# Bias Detection (AI Act Compliance)

**EU AI Act Requirement:** Credit scoring systems must be tested for bias and discrimination.

Detect Potential Bias

**AI Act Requirement:** Fairness Testing  
**Metric:** Disparate Impact (80% Rule)

In [5]:
# Approval rate by gender - is there disparate impact?
pipeline_gender_bias = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {"$cond": ["$decision.loan_approved", 1, 0]}
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {"$divide": ["$approved", "$total"]}
        }
    },
    {
        "$sort": {"approval_rate": -1}
    }
]

gender_bias = list(collection.aggregate(pipeline_gender_bias))

print("Approval Rates by Gender:")
print("=" * 50)

for group in gender_bias:
    print(f"{group['_id']}: {group['approval_rate']*100:.1f}% "
          f"({group['approved']}/{group['total']})")

Approval Rates by Gender:
Unknown: 100.0% (2/2)
Male: 66.0% (163/247)
Female: 50.6% (127/251)
